In [52]:
import pybamm
import numpy as np
import pickle

# pybamm.set_logging_level("INFO")
pybamm.set_logging_level("WARNING")

data_DIR = "../data/"

In [ ]:
vars = ["Time [s]",#0
        "Time [h]",
        "Terminal voltage [V]",
        "Current [A]",
        "X-averaged negative particle surface concentration [mol.m-3]",
        "X-averaged positive particle surface concentration [mol.m-3]",#5
        "X-averaged negative particle concentration [mol.m-3]",
        "X-averaged positive particle concentration [mol.m-3]",
        "R-averaged negative particle concentration [mol.m-3]",
        "R-averaged positive particle concentration [mol.m-3]",
        "Average negative particle concentration [mol.m-3]",#10
        "Average positive particle concentration [mol.m-3]",
        # New pybamm voltage components
        "X-averaged battery reaction overpotential [V]",
        "X-averaged battery concentration overpotential [V]",
        "X-averaged battery electrolyte ohmic losses [V]",
        "X-averaged battery solid phase ohmic losses [V]",#15
        "Battery open-circuit voltage [V]",
        "Loss of lithium to negative SEI [mol]",#17
        "X-averaged negative total SEI thickness [m]",
        "X-averaged negative electrode SEI interfacial current density [A.m-2]",
        "X-averaged negative electrode reaction overpotential [V]",
]

In [54]:
parameter_values = pybamm.ParameterValues("Mohtat2020")

def nmc_volume_change_mohtat(sto,c_s_max):
    t_change = -1.10/100*(1-sto)
    return t_change

def graphite_volume_change_mohtat(sto,c_s_max):
    stoichpoints = np.array([0,0.12,0.18,0.24,0.50,1])
    thicknesspoints = np.array([0,2.406/100,3.3568/100,4.3668/100,5.583/100,13.0635/100])
    x = [sto]
    t_change = pybamm.Interpolant(stoichpoints, thicknesspoints, x, name=None, interpolator='linear', extrapolate=True, entries_string=None)
    return t_change

par_val = {}
# Room temp
par_val[0] = [4.0312e-08,1.8157e-07,1.0776,2.3586e-09,-4.9170e-09,-1.4406e-09,4.60788219e-16,4.56607447e-19]
Temp = 25
sno = 0
parameter_values.update(
    {
        # mechanical properties
        "Positive electrode Poisson's ratio": 0.3,
        "Positive electrode Young's modulus [Pa]": 375e9,
        "Positive electrode reference concentration for free of deformation [mol.m-3]": 0,
        "Positive electrode partial molar volume [m3.mol-1]": 7.28e-7,
        "Positive electrode volume change": nmc_volume_change_mohtat,
        # Loss of active materials (LAM) model
        "Positive electrode LAM constant exponential term": 2,
        "Positive electrode critical stress [Pa]": 375e6,
        # mechanical properties
        "Negative electrode Poisson's ratio": 0.2,
        "Negative electrode Young's modulus [Pa]": 15e9,
        "Negative electrode reference concentration for free of deformation [mol.m-3]": 0,
        "Negative electrode partial molar volume [m3.mol-1]": 3.1e-6,   
        "Negative electrode volume change": graphite_volume_change_mohtat,
        # Loss of active materials (LAM) model
        "Negative electrode LAM constant exponential term": 2,
        "Negative electrode critical stress [Pa]": 60e6,
        # Other
        "Cell thermal expansion coefficient [m.K-1]": 1.48E-6,
        "Lower voltage cut-off [V]": 3.0,
        # "Negative electrode active material volume fraction": eps_n_data,
        # "Positive electrode active material volume fraction": eps_p_data,
        "Initial temperature [K]": 273.15+Temp,
        "Ambient temperature [K]": 273.15+Temp,
        "SEI kinetic rate constant [m.s-1]":  par_val[sno][6], #1.08494281e-16 , 
        "EC diffusivity [m2.s-1]": par_val[sno][7],#8.30909086e-19,
        "SEI growth activation energy [J.mol-1]": 1.87422275e+04,#1.58777981e+04,
        "Initial inner SEI thickness [m]": 0e-09,
        "Initial outer SEI thickness [m]": 5e-09,
        "SEI resistivity [Ohm.m]": 30000.0,
        "Negative electrode partial molar volume [m3.mol-1]": 7e-06,
        # Initializing Particle Concentration
        # "Initial concentration in negative electrode [mol.m-3]": x100*parameter_values["Maximum concentration in negative electrode [mol.m-3]"],
        # "Initial concentration in positive electrode [mol.m-3]": y100*parameter_values["Maximum concentration in positive electrode [mol.m-3]"]
    },
    check_already_exists=False,
)

## Charge Discharge 1 Cycle

In [55]:
# experiment
crate = "C/5"
crate_str = "Cb5"
period = "10 s"

experiment = pybamm.Experiment(
    [
        f"Charge at {crate} until 4.2V",
        "Hold at 4.2 V until C/100",
        f"Discharge at {crate} until 3V",
    ],
    period = period,
)

In [56]:
# model
model = pybamm.lithium_ion.SPM(
    {
        "SEI": "ec reaction limited",
    }
)

sim = pybamm.Simulation(
    model,
    experiment=experiment,
    parameter_values=parameter_values,
)
solution = sim.solve(initial_soc=0)
d1 = {}
for i,x in enumerate(vars):
    d1[i] = solution[x].entries
with open(data_DIR +f'spm_new_sei_1cycle_vars_{crate_str}.pickle', 'wb') as handle:
    pickle.dump(d1, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [60]:
model.variables.search("SEI")

Results for 'SEI': ['Negative SEI thickness [m]', 'X-averaged negative SEI thickness [m]', 'Negative SEI [m]', 'Negative total SEI thickness [m]', 'X-averaged negative total SEI thickness [m]', 'Positive SEI thickness [m]', 'X-averaged positive SEI thickness [m]', 'Positive SEI [m]', 'Positive total SEI thickness [m]', 'X-averaged positive total SEI thickness [m]', 'Positive electrode SEI interfacial current density [A.m-2]', 'X-averaged positive electrode SEI interfacial current density [A.m-2]', 'Negative SEI on cracks thickness [m]', 'X-averaged negative SEI on cracks thickness [m]', 'Negative SEI on cracks [m]', 'Negative total SEI on cracks thickness [m]', 'X-averaged negative total SEI on cracks thickness [m]', 'Negative electrode SEI on cracks interfacial current density [A.m-2]', 'X-averaged negative electrode SEI on cracks interfacial current density [A.m-2]', 'Positive SEI on cracks thickness [m]', 'X-averaged positive SEI on cracks thickness [m]', 'Positive SEI on cracks [m]

In [57]:
# model
model = pybamm.lithium_ion.DFN(
    {
        "SEI": "ec reaction limited",
    }
)
sim = pybamm.Simulation(
    model,
    experiment=experiment,
    parameter_values=parameter_values,
)
solution = sim.solve(initial_soc=0)
d1 = {}
for i,x in enumerate(vars):
    d1[i] = solution[x].entries
with open(data_DIR +f'dfn_new_sei_1cycle_vars_{crate_str}.pickle', 'wb') as handle:
    pickle.dump(d1, handle, protocol=pickle.HIGHEST_PROTOCOL)

## Multiple Cycles

In [61]:
# experiment
crate = "C/5"
crate_str = "Cb5"
period = "10 s"

experiment = pybamm.Experiment(
    [
        f"Charge at {crate} until 4.2V",
        "Hold at 4.2 V until C/100",
        f"Discharge at {crate} until 3V",
    ]*100,
    period = period,
)

In [62]:
# model
model1 = pybamm.lithium_ion.SPM(
    {
        "SEI": "ec reaction limited",
    }
)

sim1 = pybamm.Simulation(
    model1,
    experiment=experiment,
    parameter_values=parameter_values,
)
solution1 = sim1.solve(initial_soc=0)

In [76]:
solution1.summary_variables.get_summary_variables()

{'Time [s]': [17666.410598173115,
  18364.697929999886,
  36113.8023008926,
  53656.75759986561,
  54355.47973636706,
  72100.70864722854,
  89639.63938197464,
  90338.79654486864,
  108080.13608741459,
  125615.02794766195,
  126314.62138760969,
  144052.06020525904,
  161582.90201134337,
  162282.93117772826,
  180016.45749737,
  197543.23700112567,
  198243.70339315073,
  215973.2888208349,
  233495.97843983918,
  234196.8795746687,
  251922.54342053487,
  269441.1620561093,
  270142.49930724635,
  287864.21051398286,
  305378.7266899426,
  306080.5013143859,
  323798.25505984,
  341308.66593789705,
  342010.9709381715,
  359724.79816027556,
  377231.13437910273,
  377933.8718662685,
  395643.7626058753,
  413146.0147871651,
  413849.183113096,
  431555.13715501537,
  449053.30506211607,
  449756.9043904156,
  467458.9180234215,
  484952.9982482644,
  485657.0276921172,
  503355.10521351564,
  520845.1026499538,
  521549.5604745075,
  539243.7004432016,
  556729.6135657937,
  557434

In [77]:
sol1_sumvar = solution1.summary_variables.get_summary_variables()
with open(data_DIR +f'spm_new_sei_100cycle_vars_{crate_str}.pickle', 'wb') as handle:
    pickle.dump(sol1_sumvar, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [64]:
# model
model2 = pybamm.lithium_ion.DFN(
    {
        "SEI": "ec reaction limited",
    }
)

sim2 = pybamm.Simulation(
    model2,
    experiment=experiment,
    parameter_values=parameter_values,
)
solution2 = sim2.solve(initial_soc=0)

In [78]:
sol2_sumvar = solution2.summary_variables.get_summary_variables()
with open(data_DIR +f'dfn_new_sei_100cycle_vars_{crate_str}.pickle', 'wb') as handle:
    pickle.dump(sol2_sumvar, handle, protocol=pickle.HIGHEST_PROTOCOL)